# BONTAM Valuation Tool

In [ ]:

# %%
from datetime import date



discount_rate_table = {
    1: 0.02322,
    2: 0.02332,
    3: 0.02333,
    4: 0.02333
}

emission_date = date(2025, 1, 29) #emission date for all bonds in this code is the 29th of January 2025
tamar_start_date = date(2025, 1, 15) # average starts counting 10 business days before emission

M = 10000# Number of paths to simulated
#tuned simulation parameters
sigma = 0.1428
alpha = 0.5
theta = 0.21 # Long-term rate for CIR
kappa = 0.5  # Mean reversion for CIR

interest_simulation_type = 'CIR' # should be 'CIR' 'bk' 'hw' or 'basic'


# %%
# @title

import numpy as np
import pandas as pd
import math as mt
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.optimize import minimize_scalar, minimize
from scipy.stats import ncx2
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
import warnings
from datetime import date
import seaborn as sns
import requests
import io
from pandas.tseries.offsets import CustomBusinessDay
# Suppress specific warnings for cleaner output
warnings.filterwarnings("ignore", category=RuntimeWarning)

seed=123

ARG_HOLIDAYS_DT = pd.to_datetime(ARGENTINA_HOLIDAYS)

ARG_BD = CustomBusinessDay(holidays=ARG_HOLIDAYS_DT)

# %%
# @title
from TAMAR_api import get_tamar_daily
tamar = get_tamar_daily()
# Create observed_TAMAR_series with date index and scaled values
observed_TAMAR_series = tamar['TAMAR'][tamar_start_date:] / 100

# observed_TAMAR will be the numpy array needed for simulation functions
observed_TAMAR = observed_TAMAR_series.values

# most_recent_tamar_date is the last date of the series used for r0
most_recent_tamar_date = observed_TAMAR_series.index[-1]

# observed_TAMAR_df is simply the series wrapped in a DataFrame for plotting, so it retains the date index
observed_TAMAR_df = pd.DataFrame(observed_TAMAR_series)
from TAMAR_view.py import TAMAR_view

tamars_realized_since_setting_tamar_view = get_distance_days_252('2025-10-20', most_recent_tamar_date) #this assumes in house tamar view was created on october 20th 2025

TAMAR_view = TAMAR_view[tamars_realized_since_setting_tamar_view:] #clip the first tamar views as time moves forward in real life

# Determine the start date for TAMAR_view
# It should start the business day after the last observed TAMAR
tamar_view_start_date = observed_TAMAR_df.index[-1] + ARG_BD

# Generate business days for TAMAR_view
tamar_view_dates = pd.bdate_range(start=tamar_view_start_date, periods=len(TAMAR_view), freq=ARG_BD)

# Create a Series for TAMAR_view with the generated dates
TAMAR_view_series_indexed = pd.Series(TAMAR_view, index=tamar_view_dates)

# Concatenate observed_TAMAR_df and TAMAR_view_series_indexed into a single DataFrame for plotting
# Aligning them by converting TAMAR_view_series_indexed to a DataFrame with the same column name
combined_historical_and_view = pd.concat([observed_TAMAR_df, pd.DataFrame(TAMAR_view_series_indexed, columns=['TAMAR'])])

# Plotting with numerical index
plt.figure(figsize=(12, 6))
plt.plot(np.arange(len(observed_TAMAR_df)), observed_TAMAR_df.values, label='Historical TAMAR', color='#07203C')
plt.plot(np.arange(len(observed_TAMAR_df), len(combined_historical_and_view)), TAMAR_view_series_indexed.values, label='TAMAR View', color='#07203C', linestyle='--')
plt.title('Historical TAMAR and TAMAR View')
plt.xlabel('Business Day Number') # Changed label to reflect numerical index
plt.ylabel('TAMAR Rate')
plt.legend()
plt.grid(True)
plt.show()

# %%
# @title
# create sims
r0 = float(observed_TAMAR[-1])
T_years = (287-tamars_realized_since_setting_tamar_view)/252
N = 287-tamars_realized_since_setting_tamar_view  #  steps

from bk_model import simulate_BK_trajectories
from cir_model import simulate_cir_trajectories
from hull_white import simulate_HW_trajectories
from simple_diffusion import simulate_trajectories
basic = simulate_trajectories(sigma=sigma, r0=r0, T_years=T_years, N=N, M=M)
CIR = simulate_cir_trajectories(sigma=sigma, kappa = kappa, theta=theta, r0=r0, T_years=T_years, N=N, M=M)
hw = simulate_HW_trajectories(r0=r0, alpha=alpha, sigma=sigma, TAMAR_view=TAMAR_view, M=M, T_years=T_years, N=N)
bk = simulate_BK_trajectories(r0=r0, alpha=alpha, sigma=sigma, TAMAR_view=TAMAR_view, M=M, T_years=T_years, N=N)

if interest_simulation_type == 'CIR':
  simulation_type = CIR
elif interest_simulation_type == 'hw':
  simulation_type = hw
elif interest_simulation_type == 'bk':
  simulation_type = bk
elif interest_simulation_type == 'basic':
  simulation_type = basic
else:
 raise ValueError('please choose a valid sim type')

# Get the last date from the historical data
last_observed_date = observed_TAMAR_df.index[-1]

# Generate future business days for the simulation
# The simulation_type array has N+1 rows, where the first row is r0 (at last_observed_date).
# The subsequent N rows are the actual simulated steps.
sim_start_date_for_index = last_observed_date + ARG_BD
sim_dates = pd.bdate_range(start=sim_start_date_for_index, periods=N, freq=ARG_BD)

# Create DataFrame for simulated data with generated dates, excluding the initial r0 row
sim_df_indexed = pd.DataFrame(simulation_type[1:, :], index=sim_dates, columns=[f'Path_{i}' for i in range(M)])

# Prepare historical data for concatenation - duplicate the single column of observed_TAMAR_df M times
historical_expanded = pd.DataFrame({f'Path_{i}': observed_TAMAR_df.iloc[:, 0] for i in range(M)})

# Combine observed and simulated data
combined_df = pd.concat([historical_expanded, sim_df_indexed])
plt.rcParams["font.family"] = "Tahoma"
C3_color = (0 / 255, 49 / 255, 77 / 255)

plt.figure(figsize=(12, 6)) # Make plot larger for better visibility

# Plot using numerical index instead of datetime index
plt.plot(np.arange(len(combined_df)), combined_df.values, color = C3_color, linewidth= 1)
plt.title(f'Figure 1. {interest_simulation_type} Simulated and Historical TAMAR Paths', loc='left', fontweight='bold', fontsize=14, x=0, pad=15)
plt.xlabel('Business Days Since Issuance Accrual Average', fontsize = 10) # Changed label to reflect numerical index
plt.ylabel('TAMAR Rate', fontsize = 10)
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
plt.grid(True)
plt.show()

# %%

bond_expiry = pd.to_datetime('2026-03-16')
base_rate = 0.0225
discount_rate = discount_rate_table[1] #0.02322
trading_price = TTM26_price #138.4
TTM26_expected_tv= trading_price/(discounted_value(1,discount_rate,bond_expiry))
print(TTM26_expected_tv)
TTM26_trading_price = trading_price
bond_name = 'TTM26'
figure_number = '4'
terminal_values = get_terminal_value(bond_expiry,base_rate,simulation_type)
TTM26_discounted_vpv, TTM26_var, TTM26_non_discount_option_value, TTM26_vpv_with_floor = discounted_data(terminal_values, discount_rate, bond_expiry, trading_price, bond_name, figure_number)

# %%
bond_expiry = pd.to_datetime('2026-06-30')
base_rate = 0.0219
discount_rate = discount_rate_table[2] #0.02332
trading_price = TTJ26_price #136.8 #136.65
TTJ26_expected_tv= trading_price/(discounted_value(1,discount_rate,bond_expiry))
TTJ26_trading_price = trading_price
bond_name = 'TTJ26'
figure_number = '5'
terminal_values = get_terminal_value(bond_expiry,base_rate,simulation_type)
TTJ26_discounted_vpv, TTJ26_var,TTJ26_non_discount_option_value, TTJ26_vpv_with_floor = discounted_data(terminal_values, discount_rate, bond_expiry, trading_price, bond_name, figure_number)

# %%
bond_expiry = pd.to_datetime('2026-09-15')
base_rate = 0.0217
discount_rate = discount_rate_table[3] #0.02333
trading_price = TTS26_price #134.5 #134.75
TTS26_expected_tv= trading_price/(discounted_value(1,discount_rate,bond_expiry))
TTS26_trading_price = trading_price
bond_name = 'TTS26'
figure_number = '6'
terminal_values = get_terminal_value(bond_expiry,base_rate,simulation_type)
TTS26_discounted_vpv, TTS26_var, TTS26_non_discount_option_value, TTS26_vpv_with_floor = discounted_data(terminal_values, discount_rate, bond_expiry, trading_price, bond_name, figure_number)



bond_expiry = pd.to_datetime('2026-12-15')
base_rate = 0.0214
discount_rate = discount_rate_table[4] #0.02333
trading_price = TTD26_price #132.5 #131.9
TTD26_expected_tv= trading_price/(discounted_value(1,discount_rate,bond_expiry))
TTD26_trading_price = trading_price
bond_name = 'TTD26'
figure_number = '7'
terminal_values = get_terminal_value(bond_expiry,base_rate,simulation_type)
TTD26_discounted_vpv, TTD26_var, TTD26_non_discount_option_value, TTD26_vpv_with_floor = discounted_data(terminal_values, discount_rate, bond_expiry, trading_price, bond_name, figure_number)


# %%
bdays_till_expiry = [60, 131, 186,246]
trading_prices = [TTM26_trading_price, TTJ26_trading_price, TTS26_trading_price, TTD26_trading_price]
risk_neutral_non_discount_option = [TTM26_non_discount_option_value, TTJ26_non_discount_option_value, TTS26_non_discount_option_value, TTD26_non_discount_option_value]
risk_neutral_fair_values = [TTM26_discounted_vpv, TTJ26_discounted_vpv, TTS26_discounted_vpv, TTD26_discounted_vpv]
expected_tvs = [TTM26_expected_tv, TTJ26_expected_tv, TTS26_expected_tv, TTD26_expected_tv]
print(expected_tvs)
variances_in_payment = [TTM26_var, TTJ26_var, TTS26_var, TTD26_var]
non_discounted_terminal_values = [TTM26_vpv_with_floor,TTJ26_vpv_with_floor,TTS26_vpv_with_floor,TTD26_vpv_with_floor]
difference_in_view = [0,0,0,0]
for x in range(0,4):
  difference_in_view[x] = risk_neutral_fair_values[x] - trading_prices[x]


# %%
estimated_risk_aversion_coefficient = [0,0,0,0]
for x in range(0,4):
  estimated_risk_aversion_coefficient[x] = (non_discounted_terminal_values[x]-expected_tvs[x])/(variances_in_payment[x])

print(estimated_risk_aversion_coefficient)
optimal_risk_aversion = np.mean(estimated_risk_aversion_coefficient)
print(optimal_risk_aversion)


# %%
risk_averse_no_discount_prices = [0,0,0,0]

for x in range(0,4):
  risk_averse_no_discount_prices[x] = non_discounted_terminal_values[x]-optimal_risk_aversion*(variances_in_payment[x])



# %%



expiry_lookup_table = {
    1: '2026-03-16',
    2: '2026-06-30',
    3: '2026-09-15',
    4: '2026-12-15'
}

risk_averse_discounted_prices = [0]*4 # Initialize with 4 zeros
for x  in range(1,5):
  # Use x-1 for indexing risk_averse_no_discount_prices and risk_averse_discounted_prices (0-indexed lists)
  # Use x directly for discount_rate_table and expiry_lookup_table (1-indexed dictionaries)
  risk_averse_discounted_prices[x-1] = discounted_value(risk_averse_no_discount_prices[x-1], discount_rate_table[x], expiry_lookup_table[x])



# %%
plt.rcParams["font.family"] = "Tahoma"
threshold = 0.2760
grey_color = (165 / 255, 165 / 255, 165 / 255)
navy_color = (0 / 255, 121 / 255, 204 / 255)
C3_color = (0 / 255, 49 / 255, 77 / 255)



# %%

font_family = 'Tahoma'
data = {
    'TAMAR Observations Left to Accrue': bdays_till_expiry,
    'Trading Price': trading_prices,
    'Puente Risk-Neutral Fair Value': risk_neutral_fair_values,
    'Puente Risk-Averse Discounted Price': risk_averse_discounted_prices,
    'Bond Name': ['BONTAM Mar-26 (TTM26)', 'BONTAM JUN-26 (TTJ26)', 'BONTAM SEP-26 (TTS26)', 'BONTAM DEC-26 (TTD26)']
}
df = pd.DataFrame(data)
df = df.set_index('Bond Name')

# Create a Styler object
styled_df = df.style

# Apply header styling (blue background, white text) and data cell styling (white background, black text)
styled_df = styled_df.set_table_styles([
    {'selector': 'th',
     'props': [('background-color', '#07203C'), ('color', 'white'),('font-size','12pt'),('font-weight', 'bold'), ('font-family','Tahoma'),('text-align', 'center')]},
    {'selector': 'td',
     'props': [('background-color', 'white'), ('font-family','Tahoma'),('color', 'black')]},
    {'selector': 'th.row_heading',
     'props': [('background-color', '#D9D9D9'),('font-family','Tahoma'), ('color', 'black')]}
])

# Set precision for float numbers
styled_df = styled_df.format(precision=3)

# Display the styled DataFrame
display(styled_df)



plt.plot(bdays_till_expiry, trading_prices, color = navy_color, linestyle = '--')
plt.plot(bdays_till_expiry, risk_neutral_fair_values, color = C3_color)
plt.title('Figure 2. PUENTE BONTAMs Fair Values Based on In-House Model', loc='left', fontweight='bold', fontsize=14, x=-0.10, pad=13)
#plt.text( "Annual nominal rate (%) - Incorporating Volatility to the PUENTE Base Case for TAMAR", fontsize=9, ha='left')
plt.legend([ 'Trading Price', 'Risk-Neutral Fair Value'])
plt.xlabel('TAMAR Observations Left to Accrue', fontsize=10)
plt.ylabel('Price in ARS', fontsize = 10)
plt.grid(True)
plt.show()


plt.plot(bdays_till_expiry,risk_averse_discounted_prices, color = grey_color)
plt.plot(bdays_till_expiry, trading_prices, color = navy_color, linestyle = '--')
plt.plot(bdays_till_expiry, risk_neutral_fair_values, color = C3_color)
plt.title('Figure 5. PUENTE BONTAMs Fair Values Based on In-House Model', loc='left', fontweight='bold', fontsize=14, x=-0.10, pad=13)
#plt.text( "Annual nominal rate (%) - Incorporating Volatility to the PUENTE Base Case for TAMAR", fontsize=9, ha='left')
plt.legend(['Risk-Averse Discounted Price', 'Trading Price', 'Risk-Neutral Fair Value'])
plt.xlabel('TAMAR Observations Left to Accrue', fontsize=10)
plt.ylabel('Price in ARS', fontsize = 10)
plt.grid(True)
plt.show()


Matplotlib is building the font cache; this may take a moment.
c:\Users\gasto\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.bcra.gob.ar'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\gasto\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.bcra.gob.ar'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


KeyboardInterrupt: 